# 00 - Sample-size determination (RQ1-RQ4)

**Purpose.** Compute the minimum sample size each research question needs, using the
method that fits that question's test, *before* looking at any data. The final study
sample is the maximum of the four minimums. Later notebooks check that the real data
meets every minimum.

| RQ | Test | Method | Source |
|---|---|---|---|
| RQ1 | Which pre-departure factors influence late arrival (logistic regression) | Confidence-interval (margin-of-error) method for a proportion | Cohen (1988) |
| RQ2 | Correlation between load factor and late-arrival rate | Power analysis, Fisher z-transformation | Cohen (1988) |
| RQ3 | Cancellation rate differs across clusters | Two-proportion z-test | Fleiss, Levin, & Paik (2003) |
| RQ4 | ML vs SARIMA forecast error, paired by route | Paired-difference power analysis (Cohen's d) | Cohen (1988) |

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
from scipy.stats import norm

In [2]:
z_a = norm.ppf(1 - ALPHA / 2)   # 1.96 for alpha = .05 two-tailed
z_b = norm.ppf(POWER)           # 0.8416 for power = .80
print(f"Z(1-alpha/2) = {z_a:.4f}   Z(power) = {z_b:.4f}")

# RQ1 - proportion, margin-of-error method:  n = Z^2 p(1-p) / E^2
n1 = np.ceil(z_a**2 * RQ1_P * (1 - RQ1_P) / RQ1_E**2)

# RQ2 - correlation, Fisher z:  n = ((Z_a + Z_b) / C)^2 + 3,  C = 0.5 ln((1+r)/(1-r))
C = 0.5 * np.log((1 + RQ2_R) / (1 - RQ2_R))
n2 = np.ceil(((z_a + z_b) / C) ** 2 + 3)

# RQ3 - two proportions (per group)
pbar = (RQ3_P1 + RQ3_P2) / 2
num = (z_a * np.sqrt(2 * pbar * (1 - pbar)) + z_b * np.sqrt(RQ3_P1 * (1 - RQ3_P1) + RQ3_P2 * (1 - RQ3_P2))) ** 2
n3_group = np.ceil(num / (RQ3_P1 - RQ3_P2) ** 2)
n3_total = 2 * n3_group

# RQ4 - paired difference:  n = ((Z_a + Z_b) / d)^2
n4 = np.ceil(((z_a + z_b) / RQ4_D) ** 2)

table = pd.DataFrame([
    ["RQ1", "Proportion CI (margin of error)", f"Z={z_a:.2f}; p={RQ1_P}; E={RQ1_E}", int(n1), "flights"],
    ["RQ2", "Correlation power (Fisher z)", f"alpha={ALPHA}; power={POWER}; r={RQ2_R}", int(n2), "route-months"],
    ["RQ3", "Two-proportion z-test", f"alpha={ALPHA}; power={POWER}; p1={RQ3_P1}; p2={RQ3_P2}", int(n3_total), f"flights ({int(n3_group):,} per group)"],
    ["RQ4", "Paired difference (Cohen d)", f"alpha={ALPHA}; power={POWER}; d={RQ4_D}", int(n4), "routes"],
], columns=["RQ", "Method", "Parameters", "Minimum N", "Unit"])
final_n = int(max(n1, n2, n3_total, n4))
print(table.to_string(index=False))
print(f"\nFinal minimum study sample = max(RQ1..RQ4) = {final_n:,}")
table.to_csv(TAB / "sample_size.csv", index=False)
save_result("sample_size", {
    "z_alpha": z_a, "z_beta": z_b, "rq1": int(n1), "rq2": int(n2), "rq2_C": float(C),
    "rq3_per_group": int(n3_group), "rq3_total": int(n3_total), "rq3_numerator": float(num),
    "rq3_pbar": float(pbar), "rq3_inner1": float(2 * pbar * (1 - pbar)),
    "rq3_inner2": float(RQ3_P1 * (1 - RQ3_P1) + RQ3_P2 * (1 - RQ3_P2)),
    "rq3_term1": float(z_a * np.sqrt(2 * pbar * (1 - pbar))),
    "rq3_term2": float(z_b * np.sqrt(RQ3_P1 * (1 - RQ3_P1) + RQ3_P2 * (1 - RQ3_P2))),
    "rq3_raw_n": float(num / (RQ3_P1 - RQ3_P2) ** 2), "rq1_raw_n": float(z_a**2 * RQ1_P * (1 - RQ1_P) / RQ1_E**2),
    "rq2_raw_n": float(((z_a + z_b) / C) ** 2 + 3), "rq4_raw_n": float(((z_a + z_b) / RQ4_D) ** 2),
    "rq4": int(n4), "final_n": final_n})

Z(1-alpha/2) = 1.9600   Z(power) = 0.8416
 RQ                          Method                                Parameters  Minimum N                      Unit
RQ1 Proportion CI (margin of error)                     Z=1.96; p=0.5; E=0.01       9604                   flights
RQ2    Correlation power (Fisher z)              alpha=0.05; power=0.8; r=0.1        783              route-months
RQ3           Two-proportion z-test alpha=0.05; power=0.8; p1=0.015; p2=0.025       6152 flights (3,076 per group)
RQ4     Paired difference (Cohen d)              alpha=0.05; power=0.8; d=0.3         88                    routes

Final minimum study sample = max(RQ1..RQ4) = 9,604


{'z_alpha': 1.959963984540054,
 'z_beta': 0.8416212335729143,
 'rq1': 9604,
 'rq2': 783,
 'rq2_C': 0.10033534773107562,
 'rq3_per_group': 3076,
 'rq3_total': 6152,
 'rq3_numerator': 0.30755816558915616,
 'rq3_pbar': 0.02,
 'rq3_inner1': 0.0392,
 'rq3_inner2': 0.039150000000000004,
 'rq3_term1': 0.3880530708179098,
 'rq3_term2': 0.1665261981681894,
 'rq3_raw_n': 3075.5816558915603,
 'rq1_raw_n': 9603.647051735314,
 'rq2_raw_n': 782.6501276111056,
 'rq4_raw_n': 87.20977482610098,
 'rq4': 88,
 'final_n': 9604}

**Reading the result.** RQ1 needs the largest sample (9,604 flights) because its
margin-of-error requirement is the tightest, so it sets the study minimum. BTS publishes
every reporting-carrier flight (a census, not a sample), so these minimums are met by
a wide margin; notebook 01 documents the actual counts.